# 第3回実験：Metric Outdoor Small 20エポック計画

20エポック総期間の学習率スケジュールを使い、まず5エポックで安全停止します。第2回の `source_v4/` を新しい実験専用フォルダーへ複製し、完全checkpoint、乱数状態の保存・復元、エポック境界停止、固定されたvalidation形状サンプル集計をCPUで検証してからGPUへ切り替えます。

独立test・過去52組は読みません。Hugging Faceからの取得、データ再取得、既存checkpointの上書き、Compute Units購入は行いません。GPUは無料割当だけを使用し、事前計測がpassしたときだけ学習へ進みます。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, math, shutil, compileall, sys
import torch
from google.colab import drive

drive.mount('/content/drive')
assert not torch.cuda.is_available(), 'CPU検証セルです。GPUへ切り替える前にCPUランタイムで実行してください。'
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP2 = ROOT / 'experiments/metric_shape_20261013T123006472484Z'
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
SOURCE_V4 = EXP2 / 'source_v4'
SOURCE_FULL = EXP3 / 'source_full_v1'
SOURCE_MARKER = EXP3 / 'source_build.json'
PLAN = ROOT / 'uav_metric_finetuning/config/verified_plan'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
assert (EXP2 / 'report_ja.md').is_file() and (EXP2 / 'analysis/shape_summary.json').is_file()
assert (SOURCE_V4 / 'train.py').is_file() and (SOURCE_V4 / 'config/scene_split_plan.json').is_file()
assert (PLAN / 'train.csv').is_file() and (PLAN / 'validation.csv').is_file()
assert EXP3.is_dir(), f'Drive上の新規実験フォルダーが見つかりません: {EXP3}'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def source_inventory(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*'))
            if p.is_file() and '__pycache__' not in p.parts}

if SOURCE_MARKER.is_file():
    marker = json.loads(SOURCE_MARKER.read_text(encoding='utf-8'))
    assert marker['source_v4_path'] == str(SOURCE_V4)
    assert marker['patched_files'] == source_inventory(SOURCE_FULL), 'source_full_v1が保存後に変更されています。安全のため停止します。'
    print('既存の検証済みsource_full_v1を再利用:', SOURCE_FULL)
else:
    assert not SOURCE_FULL.exists(), f'不完全または未確認のsource_full_v1を保護して停止します: {SOURCE_FULL}'
    shutil.copytree(SOURCE_V4, SOURCE_FULL, ignore=shutil.ignore_patterns('__pycache__'))

    def replace_once(text, old, new, label):
        count = text.count(old)
        if count != 1:
            raise RuntimeError(f'{label}: patch anchor count={count}; copied source is unchanged and training must not start')
        return text.replace(old, new, 1)

    train_path = SOURCE_FULL / 'train.py'
    train = train_path.read_text(encoding='utf-8')
    rng_helpers = """\n\ndef capture_rng_state() -> dict[str, Any]:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state: dict[str, Any]) -> None:
    required = {"python", "numpy", "torch_cpu", "torch_cuda"}
    if not required.issubset(state):
        raise ValueError("完全checkpointに必要な乱数状態がありません")
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if state["torch_cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA乱数状態を復元できません。GPUランタイムへ接続してください。")
        torch.cuda.set_rng_state_all(state["torch_cuda"])


def should_pause_after_epoch(epoch_number: int, total_epochs: int, stop_after_epoch: int | None) -> bool:
    return stop_after_epoch is not None and epoch_number >= stop_after_epoch and epoch_number < total_epochs
"""
    train = replace_once(train, '\n\ndef check_split_isolation(', rng_helpers + '\n\ndef check_split_isolation(', 'RNG helpers')
    train = replace_once(
        train,
        '                "scaler": scaler.state_dict(),\n                "best_validation_rmse_m":',
        '                "scaler": scaler.state_dict(),\n                "rng_state": capture_rng_state(),\n                "best_validation_rmse_m":',
        'checkpoint RNG save',
    )
    train = replace_once(
        train,
        '    scaler.load_state_dict(state["scaler"])\n    return state',
        '    scaler.load_state_dict(state["scaler"])\n    restore_rng_state(state.get("rng_state", {}))\n    return state',
        'checkpoint RNG restore',
    )
    train = replace_once(train, '        seed_all(args.seed + epoch_index)\n', '', 'epoch RNG continuity')
    train = replace_once(
        train,
        '    parser.add_argument("--resume", action="store_true", help="Resume only this run directory and identical configuration")\n    args = parser.parse_args()',
        '    parser.add_argument("--resume", action="store_true", help="Resume only this run directory and identical configuration")\n'
        '    parser.add_argument("--stop-after-epoch", type=int, default=None, help="Pause cleanly at this completed epoch; excluded from the 20-epoch schedule signature")\n'
        '    args = parser.parse_args()\n'
        '    if args.stop_after_epoch is not None and not 1 <= args.stop_after_epoch <= args.epochs:\n'
        '        parser.error("stop-after-epoch must be between 1 and the fixed total epoch count")',
        'session stop option',
    )
    old_status = """        write_json_atomic(run_dir / "run_status.json", {
            "status": "completed" if epoch_number == args.epochs else "running",
            "completed_epochs": completed_epoch,
            "remaining_epochs": max(args.epochs - completed_epoch, 0),
            "best_validation_rmse_m": best_rmse if math.isfinite(best_rmse) else None,
            "best_epoch": best_epoch,
        })
    else:
        write_json_atomic(run_dir / "run_status.json", {"""
    new_status = """        should_pause = should_pause_after_epoch(epoch_number, args.epochs, args.stop_after_epoch)
        status = "completed" if epoch_number == args.epochs else ("paused_after_epoch" if should_pause else "running")
        write_json_atomic(run_dir / "run_status.json", {
            "status": status,
            "completed_epochs": completed_epoch,
            "remaining_epochs": max(args.epochs - completed_epoch, 0),
            "best_validation_rmse_m": best_rmse if math.isfinite(best_rmse) else None,
            "best_epoch": best_epoch,
        })
        if should_pause:
            print(f"Paused safely after epoch {completed_epoch}/{args.epochs}; scheduler remains configured for the full schedule.")
            break
    else:
        write_json_atomic(run_dir / "run_status.json", {"""
    train = replace_once(train, old_status, new_status, 'epoch-boundary pause')
    train_path.write_text(train, encoding='utf-8')

    metrics_path = SOURCE_FULL / 'metrics.py'
    metrics = metrics_path.read_text(encoding='utf-8')
    sampled_helpers = """\n\nclass SampledDeltaAccumulator:
    def __init__(self) -> None:
        self.total = [0, 0.0, 0.0, 0.0, 0, 0]
        self.small = [0, 0.0, 0.0, 0.0, 0, 0]
        self.large = [0, 0.0, 0.0, 0.0, 0, 0]

    @staticmethod
    def _add(total, gt_delta, pred_delta):
        if gt_delta.size == 0:
            return
        total[0] += int(gt_delta.size)
        total[1] += float(np.abs(pred_delta - gt_delta).sum())
        total[2] += float(np.abs(gt_delta).sum())
        total[3] += float(np.abs(pred_delta).sum())
        nonzero = np.abs(gt_delta) > 1e-6
        total[4] += int(nonzero.sum())
        total[5] += int(((gt_delta[nonzero] * pred_delta[nonzero]) > 0).sum())

    def update(self, gt_delta, pred_delta) -> None:
        self._add(self.total, gt_delta, pred_delta)
        absolute = np.abs(gt_delta)
        self._add(self.small, gt_delta[absolute < 0.05], pred_delta[absolute < 0.05])
        self._add(self.large, gt_delta[absolute >= 0.20], pred_delta[absolute >= 0.20])

    @staticmethod
    def _result(values):
        n, error_sum, gt_abs_sum, pred_abs_sum, nonzero, sign_correct = values
        return {
            "pair_count": n,
            "delta_mae_m": error_sum / n if n else None,
            "mean_abs_gt_delta_m": gt_abs_sum / n if n else None,
            "mean_abs_pred_delta_m": pred_abs_sum / n if n else None,
            "amplitude_ratio": pred_abs_sum / gt_abs_sum if gt_abs_sum > 0 else None,
            "nonzero_gt_pair_count": nonzero,
            "sign_match_count": sign_correct,
            "sign_match_rate": sign_correct / nonzero if nonzero else None,
        }

    def result(self):
        return {
            "all_sampled_pairs": self._result(self.total),
            "gt_abs_delta_0_5cm": self._result(self.small),
            "gt_abs_delta_ge_20cm": self._result(self.large),
        }


def sampled_local_shape(prediction, target, valid, image_index, crop, view, stride=14, candidates_per_axis=100000):
    prediction = prediction.detach().float().cpu().numpy()
    target = target.detach().float().cpu().numpy()
    valid = valid.detach().cpu().numpy().astype(bool, copy=False)
    height, width = target.shape
    rng = np.random.default_rng(42 + image_index * 10000 + stride)
    accumulators = {}
    candidates = 0
    valid_pairs = 0
    for axis in (0, 1):
        h = height - stride if axis == 0 else height
        w = width - stride if axis == 1 else width
        count = min(candidates_per_axis, h * w)
        flat = rng.choice(h * w, size=count, replace=False)
        y, x = flat // w, flat % w
        y1 = y + (stride if axis == 0 else 0)
        x1 = x + (stride if axis == 1 else 0)
        pair_valid = valid[y, x] & valid[y1, x1]
        pair_valid &= np.isfinite(prediction[y, x]) & np.isfinite(prediction[y1, x1])
        pair_valid &= (prediction[y, x] > 0) & (prediction[y1, x1] > 0)
        y, x, y1, x1 = y[pair_valid], x[pair_valid], y1[pair_valid], x1[pair_valid]
        gt_delta = (target[y1, x1] - target[y, x]).astype(np.float64, copy=False)
        pred_delta = (prediction[y1, x1] - prediction[y, x]).astype(np.float64, copy=False)
        edge = np.minimum.reduce([y, x, height - 1 - y, width - 1 - x, y1, x1, height - 1 - y1, width - 1 - x1])
        selectors = {
            "all": np.ones(edge.shape, dtype=bool),
            "border_0_128px": edge < 128,
            "border_128_512px": (edge >= 128) & (edge < 512),
            "interior_ge_512px": edge >= 512,
            f"view={view}": np.ones(edge.shape, dtype=bool),
            f"crop={crop}": np.ones(edge.shape, dtype=bool),
        }
        for name, select in selectors.items():
            accumulator = accumulators.setdefault(name, SampledDeltaAccumulator())
            accumulator.update(gt_delta[select], pred_delta[select])
        candidates += count
        valid_pairs += int(pair_valid.sum())
    return {
        "method": "100000 candidate pairs per image/axis without replacement; seed=42+image_index*10000+14; shared GT-valid coordinates",
        "candidate_positions": candidates,
        "valid_sampled_pairs": valid_pairs,
        "groups": {name: accumulator.result() for name, accumulator in sorted(accumulators.items())},
    }
"""
    metrics = replace_once(metrics, '\n\n@torch.inference_mode()\ndef evaluate_model(', sampled_helpers + '\n\n@torch.inference_mode()\ndef evaluate_model(', 'sampled validation metrics')
    metrics = replace_once(
        metrics,
        '    crop_accumulators: dict[str, MetricAccumulator] = {}\n    for batch in dataloader:',
        '    crop_accumulators: dict[str, MetricAccumulator] = {}\n'
        '    view_accumulators: dict[str, MetricAccumulator] = {}\n'
        '    crop_view_accumulators: dict[str, MetricAccumulator] = {}\n'
        '    sampled_rows = []\n'
        '    for image_index, batch in enumerate(dataloader):',
        'grouped validation metrics',
    )
    metrics = replace_once(
        metrics,
        '        crop_accumulators.setdefault(crop, MetricAccumulator()).update(prediction, masked_target)\n    result = accumulator.result()\n    result["per_crop"] = {crop: values.result() for crop, values in sorted(crop_accumulators.items())}\n    return result',
        '        crop_accumulators.setdefault(crop, MetricAccumulator()).update(prediction, masked_target)\n'
        '        view = str(batch["view"][0])\n'
        '        view_accumulators.setdefault(view, MetricAccumulator()).update(prediction, masked_target)\n'
        '        crop_view_accumulators.setdefault(f"{crop}|{view}", MetricAccumulator()).update(prediction, masked_target)\n'
        '        sampled_rows.append(sampled_local_shape(prediction, target, valid, image_index, crop, view))\n'
        '    result = accumulator.result()\n'
        '    result["per_crop"] = {crop: values.result() for crop, values in sorted(crop_accumulators.items())}\n'
        '    result["per_view"] = {view: values.result() for view, values in sorted(view_accumulators.items())}\n'
        '    result["per_crop_view"] = {group: values.result() for group, values in sorted(crop_view_accumulators.items())}\n'
        '    sampled_aggregates = {}\n'
        '    for sampled in sampled_rows:\n'
        '        for group, value in sampled["groups"].items():\n'
        '            target_group = sampled_aggregates.setdefault(group, SampledDeltaAccumulator())\n'
        '            # Merge exact sums by reconstructing the sampled sufficient statistics from each group result.\n'
        '            for field, key in (("all_sampled_pairs", "total"), ("gt_abs_delta_0_5cm", "small"), ("gt_abs_delta_ge_20cm", "large")):\n'
        '                row = value[field]\n'
        '                sums = getattr(target_group, key)\n'
        '                n = row["pair_count"]\n'
        '                if n:\n'
        '                    sums[0] += n\n'
        '                    sums[1] += row["delta_mae_m"] * n\n'
        '                    sums[2] += row["mean_abs_gt_delta_m"] * n\n'
        '                    sums[3] += row["mean_abs_pred_delta_m"] * n\n'
        '                    sums[4] += row["nonzero_gt_pair_count"]\n'
        '                    sums[5] += row["sign_match_count"]\n'
        '    result["sampled_local_shape"] = {\n'
        '        "method": sampled_rows[0]["method"] if sampled_rows else None,\n'
        '        "candidate_positions": sum(row["candidate_positions"] for row in sampled_rows),\n'
        '        "valid_sampled_pairs": sum(row["valid_sampled_pairs"] for row in sampled_rows),\n'
        '        "groups": {name: values.result() for name, values in sorted(sampled_aggregates.items())},\n'
        '    }\n'
        '    return result',
        'assemble validation metrics',
    )
    metrics_path.write_text(metrics, encoding='utf-8')
    assert compileall.compile_dir(str(SOURCE_FULL), quiet=1, force=True)
    marker = {
        'source_v4_path': str(SOURCE_V4),
        'source_v4_sha256': source_inventory(SOURCE_V4),
        'patched_files': source_inventory(SOURCE_FULL),
        'changes': ['full-run-only safe epoch pause', 'Python/NumPy/CPU Torch/CUDA RNG checkpoint save and restore', 'per-epoch full and common sampled local-shape validation metrics'],
        'tested_before_gpu': False,
    }
    SOURCE_MARKER.write_text(json.dumps(marker, indent=2), encoding='utf-8')
    print('source_v4を保護したまま新run用コピーを作成し、構文検査しました:', SOURCE_FULL)

with (PLAN / 'train.csv').open(encoding='utf-8-sig', newline='') as f:
    train_rows = list(csv.DictReader(f))
with (PLAN / 'validation.csv').open(encoding='utf-8-sig', newline='') as f:
    validation_rows = list(csv.DictReader(f))
assert len(train_rows) == 384 and all(r['split'] == 'train' for r in train_rows)
assert len(validation_rows) == 128 and all(r['split'] == 'validation' for r in validation_rows)
assert not RUN.exists(), f'新run名が既に存在します。上書きせず確認してください: {RUN}'
print(json.dumps({'phase': 'cpu_source_preparation', 'train_pairs': len(train_rows), 'validation_pairs': len(validation_rows), 'source_full_v1': str(SOURCE_FULL), 'run_path_available': not RUN.exists(), 'independent_test_read': False, 'gpu_training_started': False}, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
import json, random, tempfile
import numpy as np
import torch
import torch.nn.functional as F
from safetensors.torch import save_file

assert not torch.cuda.is_available(), 'CPUでcheckpoint再開の単体確認を行います。'
sys.path.insert(0, str(SOURCE_FULL))
from train import save_checkpoint, restore_checkpoint, seed_all, scaled_optimizer_step, should_pause_after_epoch
from metrics import evaluate_model

class TinyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(4, 2)
    def forward(self, x):
        return self.linear(x)
    def save_pretrained(self, folder, safe_serialization=True):
        folder = Path(folder)
        save_file({k: v.detach().cpu().contiguous() for k, v in self.state_dict().items()}, str(folder / 'model.safetensors'))
        (folder / 'config.json').write_text('{}', encoding='utf-8')

def components():
    model = TinyModel()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=4, power=0.9)
    scaler = torch.amp.GradScaler('cuda', enabled=False)
    return model, optimizer, scheduler, scaler

def next_step(model, optimizer, scheduler, scaler):
    optimizer.zero_grad(set_to_none=True)
    x = torch.randn(8, 4)
    y = torch.randn(8, 2)
    noise = random.random() + float(np.random.random())
    loss = ((model(x) - y) ** 2).mean() + noise * 1e-4
    scaler.scale(loss).backward()
    assert scaled_optimizer_step(optimizer, scheduler, scaler)
    optimizer.zero_grad(set_to_none=True)
    return noise, x, y, float(loss.detach())

with tempfile.TemporaryDirectory(prefix='trial3_resume_cpu_') as temp:
    run = Path(temp)
    seed_all(2026)
    model_a, opt_a, sched_a, scaler_a = components()
    next_step(model_a, opt_a, sched_a, scaler_a)
    checkpoint = save_checkpoint(run, 1, model_a, opt_a, sched_a, scaler_a, 1.0, 1)
    continuation_a = next_step(model_a, opt_a, sched_a, scaler_a)
    model_b, opt_b, sched_b, scaler_b = components()
    restored = restore_checkpoint(checkpoint, model_b, opt_b, sched_b, scaler_b, torch.device('cpu'))
    assert restored['completed_epoch'] == 1 and 'rng_state' in restored
    continuation_b = next_step(model_b, opt_b, sched_b, scaler_b)
    assert continuation_a[0] == continuation_b[0]
    torch.testing.assert_close(continuation_a[1], continuation_b[1], rtol=0, atol=0)
    torch.testing.assert_close(continuation_a[2], continuation_b[2], rtol=0, atol=0)
    torch.testing.assert_close(model_a.linear.weight, model_b.linear.weight, rtol=0, atol=0)
    torch.testing.assert_close(model_a.linear.bias, model_b.linear.bias, rtol=0, atol=0)
    assert sched_a.state_dict() == sched_b.state_dict()
    assert should_pause_after_epoch(5, 20, 5)
    assert not should_pause_after_epoch(5, 20, 10)
    assert not should_pause_after_epoch(20, 20, 20)

# The fixed validation sample extension is exercised on a tiny synthetic CPU-only pair.
target = 2.0 + torch.arange(64, dtype=torch.float32)[None, :] * 0.02 + torch.arange(64, dtype=torch.float32)[:, None] * 0.01
small = F.interpolate(target[None, None], size=(8, 8), mode='bilinear', align_corners=True)
pixels = small.expand(1, 3, 8, 8).contiguous()
class DummyDepth(torch.nn.Module):
    def forward(self, pixel_values):
        return type('Out', (), {'predicted_depth': pixel_values[:, 0]})()
sample_result = evaluate_model(DummyDepth(), [{'pixel_values': pixels, 'original_depth': target.unsqueeze(0), 'crop': ['corn'], 'view': ['Nadir']}], torch.device('cpu'), 65.0)
assert sample_result['valid_pixels'] == 64 * 64
assert sample_result['local_pair_count'] > 0
assert sample_result['sampled_local_shape']['groups']['crop=corn']['all_sampled_pairs']['pair_count'] > 0
assert sample_result['sampled_local_shape']['groups']['view=Nadir']['gt_abs_delta_ge_20cm']['pair_count'] > 0

train_steps_per_epoch = math.ceil(math.ceil(384 / 1) / 4)
total_scheduler_steps = train_steps_per_epoch * 20
assert train_steps_per_epoch == 96 and total_scheduler_steps == 1920
cpu_result = {
    'status': 'pass',
    'tested': ['complete checkpoint hashes', 'optimizer/scheduler/scaler restore', 'Python/NumPy/Torch RNG continuation equality', 'safe epoch 5 pause target while total epochs remain 20', 'sampled local-shape validation output and crop/view groups'],
    'train_steps_per_epoch': train_steps_per_epoch,
    'scheduler_total_optimizer_steps_for_20_epochs': total_scheduler_steps,
    'validation_sample_candidates_per_image_axis': 100000,
    'cuda_used': False,
}
test_path = EXP3 / 'cpu_resume_smoke.json'
if test_path.exists():
    prior = json.loads(test_path.read_text(encoding='utf-8'))
    assert prior.get('status') == 'pass'
else:
    test_path.write_text(json.dumps(cpu_result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(cpu_result | {'saved_to': str(test_path)}, ensure_ascii=False, indent=2))

## GPUへ切り替える前の確認

CPU検証がpassした後、Colabで**ハードウェアアクセラレータを無料GPU**に切り替え、新しいランタイムへ接続して、このセルを最初に実行してください。GPU名、空きVRAM、Drive残量、ローカルMetric Outdoor cacheを記録し、`model_max_depth=80m` と `GT max_depth=65m` を分けた合成step事前計測を実行します。計測がpassしない場合は学習セルへ進みません。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, time
import torch
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'CUDAの新しいランタイムが必要です。無料GPUが割り当てられなければ停止してください。'
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
SOURCE_FULL = EXP3 / 'source_full_v1'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
PLAN = ROOT / 'uav_metric_finetuning/config/verified_plan'
assert json.loads((EXP3 / 'cpu_resume_smoke.json').read_text())['status'] == 'pass'
assert not RUN.exists(), f'新runの保存先が既に存在します。上書きしません: {RUN}'
import psutil
active = []
for process in psutil.process_iter(['pid', 'cmdline']):
    cmd = ' '.join(process.info.get('cmdline') or [])
    if 'train.py' in cmd and 'metric_outdoor_full_20261014_s42_518_cap80_gt65' in cmd:
        active.append({'pid': process.info['pid'], 'cmd': cmd[:300]})
assert not active, f'同一runの学習プロセスを検出しました: {active}'

# Strict offline resolution: this cell never downloads model weights.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from huggingface_hub import snapshot_download
sys.path.insert(0, str(SOURCE_FULL))
from modeling import METRIC_OUTDOOR_MODEL_ID, METRIC_OUTDOOR_MODEL_REVISION
cache_dir = ROOT / 'model_cache'
metric_model = snapshot_download(repo_id=METRIC_OUTDOOR_MODEL_ID, revision=METRIC_OUTDOOR_MODEL_REVISION,
                                cache_dir=str(cache_dir), local_files_only=True,
                                allow_patterns=['config.json', '*.safetensors'])
metric_model = Path(metric_model)
assert (metric_model / 'model.safetensors').is_file() and (metric_model / 'config.json').is_file()
original_config = json.loads((metric_model / 'config.json').read_text(encoding='utf-8'))
assert float(original_config['max_depth']) == 80.0, original_config.get('max_depth')

gpu_name = torch.cuda.get_device_name(0)
free_vram, total_vram = torch.cuda.mem_get_info(0)
drive_free = shutil.disk_usage(ROOT).free
checkpoint_bytes = 99_173_660 + 197_927_323 + 1_690 + 321
full_checkpoint_estimate = checkpoint_bytes * 21  # epoch 0 plus 20 completed epochs, with no pruning
assert drive_free > full_checkpoint_estimate + 2 * 1024**3, 'Drive空き容量が20epoch分の完全checkpoint見積り+2GiBを下回ります。'
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
preflight_dir = EXP3 / 'preflight'
preflight_dir.mkdir(parents=True, exist_ok=True)
preflight_path = preflight_dir / f'metric_outdoor_res518_cap80_gt65_{stamp}.json'
cmd = [sys.executable, '-u', str(SOURCE_FULL / 'memory_preflight.py'),
       '--initialization', 'metric_outdoor', '--metric-model', str(metric_model),
       '--image-size', '518', '--batch-size', '1', '--max-depth', '65',
       '--model-max-depth', '80', '--amp', '--output', str(preflight_path)]
print(json.dumps({'gpu': gpu_name, 'free_vram_bytes_before_preflight': free_vram,
                  'total_vram_bytes': total_vram, 'drive_free_bytes': drive_free,
                  'estimated_full_checkpoint_bytes': full_checkpoint_estimate,
                  'model_snapshot': str(metric_model), 'model_output_cap_m': 80,
                  'gt_mask_max_m': 65, 'preflight_command': cmd}, ensure_ascii=False, indent=2))
preflight = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
print(preflight.stdout[-6000:])
if preflight.stderr:
    print(preflight.stderr[-3000:])
assert preflight.returncode == 0 and preflight_path.is_file(), 'VRAM事前計測が成功しませんでした。学習は開始しません。'
preflight_result = json.loads(preflight_path.read_text(encoding='utf-8'))
assert preflight_result.get('status') == 'pass', preflight_result
assert preflight_result['initialization']['max_depth_m'] == 80.0
preflight_record = {'status': preflight_result['status'], 'gpu': gpu_name,
                    'free_vram_before_bytes': free_vram, 'total_vram_bytes': total_vram,
                    'model_output_cap_m': 80, 'gt_mask_max_m': 65,
                    'snapshot': str(metric_model), 'report': str(preflight_path),
                    'completed_utc': datetime.now(timezone.utc).isoformat(),
                    'compute_policy': 'free GPU only; no Compute Units, purchase, upgrade, or limit bypass'}
record_path = EXP3 / 'gpu_preflight_verified.json'
record_path.write_text(json.dumps(preflight_record, ensure_ascii=False, indent=2), encoding='utf-8')
print('GPU_PREFLIGHT_PASS', json.dumps(preflight_record, ensure_ascii=False))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, compileall
from transformers import AutoConfig
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
SOURCE_FULL = EXP3 / 'source_full_v1'
modeling_path = SOURCE_FULL / 'modeling.py'
source = modeling_path.read_text(encoding='utf-8')
old = 'model.config.max_depth = float(max_depth)'
new = 'model.config.max_depth = int(max_depth)'
if old in source:
    source = source.replace(old, new, 1)
else:
    assert new in source, '期待した設定行を確認できません。学習は開始しません。'
modeling_path.write_text(source, encoding='utf-8')
assert compileall.compile_file(str(modeling_path), quiet=1)
metric_snapshot = ROOT / 'model_cache/models--depth-anything--Depth-Anything-V2-Metric-Outdoor-Small-hf/snapshots/fd2c22027eaf20374204f14099b8341e1925ad39'
cfg = AutoConfig.from_pretrained(str(metric_snapshot), local_files_only=True)
cfg.max_depth = int(80.0)
assert type(cfg.max_depth) is int and cfg.max_depth == 80
cfg.max_depth = int(65.0)
assert type(cfg.max_depth) is int and cfg.max_depth == 65
record = {'status':'pass','change':'strict HubConfig max_depth accepts int; preserve 80m model output cap and 65m GT mask cap','source_file':str(modeling_path),'tested_config_type':type(cfg).__name__,'cpu_config_assignment':{'80m':cfg.__class__.max_depth if hasattr(cfg.__class__,'max_depth') else 'verified through instance assignment','65m':cfg.max_depth},'modified_utc':datetime.now(timezone.utc).isoformat(),'cuda_used':False}
(EXP3 / 'preflight_config_type_fix.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
from pathlib import Path
from datetime import datetime, timezone
import json, compileall
from transformers import AutoConfig
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
SOURCE_FULL = EXP3 / 'source_full_v1'
modeling_path = SOURCE_FULL / 'modeling.py'
source = modeling_path.read_text(encoding='utf-8')
old = 'model.config.max_depth = float(max_depth)'
new = 'model.config.max_depth = int(max_depth)'
if old in source:
    source = source.replace(old, new, 1)
else:
    assert new in source, '期待した設定行を確認できません。学習は開始しません。'
modeling_path.write_text(source, encoding='utf-8')
assert compileall.compile_file(str(modeling_path), quiet=1)
metric_snapshot = ROOT / 'model_cache/models--depth-anything--Depth-Anything-V2-Metric-Outdoor-Small-hf/snapshots/fd2c22027eaf20374204f14099b8341e1925ad39'
cfg = AutoConfig.from_pretrained(str(metric_snapshot), local_files_only=True)
cfg.max_depth = int(80.0)
cap80 = {'value_m':cfg.max_depth, 'type':type(cfg.max_depth).__name__}
assert cap80 == {'value_m':80, 'type':'int'}
cfg.max_depth = int(65.0)
gt65 = {'value_m':cfg.max_depth, 'type':type(cfg.max_depth).__name__}
assert gt65 == {'value_m':65, 'type':'int'}
record = {'status':'pass','change':'strict HubConfig max_depth is integer typed; model output remains 80m and GT mask remains 65m','source_file':str(modeling_path),'tested_config_type':type(cfg).__name__,'cpu_config_assignment':{'model_output_cap':cap80,'gt_mask_cap':gt65},'modified_utc':datetime.now(timezone.utc).isoformat(),'cuda_used':False}
(EXP3 / 'preflight_config_type_fix.json').write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(record, ensure_ascii=False, indent=2))

## 512組をDriveからローカルSSDへコピーし、最初の5エポックを開始

このセルはGPU事前計測のpassを必須条件にします。512組の既存DriveデータだけをColabの一時SSDへコピーし、train 384 / validation 128、転送対象の合計7,850,330,776 bytesを確認します。その後、Metric Outdoor Smallの元80m checkpointから、総20エポックの新規full runを開始し、5エポック終了時にcheckpointを保存して一時停止します。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, math, os, shutil, subprocess, sys, time
import torch, psutil
assert torch.cuda.is_available()
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
SOURCE_FULL = EXP3 / 'source_full_v1'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
PLAN = ROOT / 'uav_metric_finetuning/config/verified_plan'
RGB_DRIVE, DEPTH_DRIVE = ROOT / 'data/uav3dcrop_rgb', ROOT / 'data/uav3dcrop_depth'
LOCAL = Path('/content/uav3dcrop')
RGB_LOCAL, DEPTH_LOCAL = LOCAL / 'rgb', LOCAL / 'depth'
preflight_path = EXP3 / 'gpu_preflight_verified.json'
assert json.loads(preflight_path.read_text())['status'] == 'pass'
assert not RUN.exists(), f'新runが既に存在します。二重起動しません: {RUN}'

rows = []
for split in ('train', 'validation'):
    with (PLAN / f'{split}.csv').open(encoding='utf-8-sig', newline='') as f:
        split_rows = list(csv.DictReader(f))
    expected_count = 384 if split == 'train' else 128
    assert len(split_rows) == expected_count and all(r['split'] == split for r in split_rows)
    rows.extend(split_rows)
planned_bytes = sum(int(r['image_size_bytes']) + int(r['depth_size_bytes']) for r in rows)
assert len(rows) == 512 and planned_bytes == 7_850_330_776, (len(rows), planned_bytes)
local_free = shutil.disk_usage('/content').free
assert local_free > planned_bytes + 2 * 1024**3, 'Colab一時SSD空き容量がデータコピーに不足しています。'
stage_log = EXP3 / 'stage_retry_attempt_02.log'
stage_cmd = [sys.executable, '-u', str(SOURCE_FULL / 'stage_selected_data.py'),
             '--plan-dir', str(PLAN), '--source-rgb-root', str(RGB_DRIVE),
             '--source-depth-root', str(DEPTH_DRIVE), '--target-rgb-root', str(RGB_LOCAL),
             '--target-depth-root', str(DEPTH_LOCAL), '--splits', 'train', 'validation', '--stage']
stage = subprocess.CompletedProcess(stage_cmd, 0, 'Reusing prior verified local stage', '') if (EXP3 / 'data_stage_verified.json').is_file() and json.loads((EXP3 / 'data_stage_verified.json').read_text()).get('status') == 'pass' else subprocess.run(stage_cmd, capture_output=True, text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
stage_log.write_text(stage.stdout + '\n' + stage.stderr, encoding='utf-8')
print(stage.stdout[-4000:])
assert stage.returncode == 0, 'Driveからの既存データのローカルコピーに失敗しました。学習は開始しません。'
sys.path.insert(0, str(SOURCE_FULL))
from runtime_state import dataset_status
local_status = dataset_status(PLAN, RGB_LOCAL, DEPTH_LOCAL, ('train', 'validation'))
assert local_status['complete'] and local_status['complete_pairs'] == 512
assert local_status['present_files'] == 1024 and local_status['planned_bytes'] == 7_850_330_776
stage_result = {'status': 'pass', 'train_pairs': 384, 'validation_pairs': 128,
                'files': local_status['present_files'], 'bytes': local_status['planned_bytes'],
                'source': 'existing Google Drive data; no external or Hugging Face data download',
                'local_status': local_status, 'completed_utc': datetime.now(timezone.utc).isoformat()}
(EXP3 / 'data_stage_verified_attempt_02.json').write_text(json.dumps(stage_result, ensure_ascii=False, indent=2), encoding='utf-8')

preflight = json.loads(preflight_path.read_text(encoding='utf-8'))
metric_model = Path(preflight['snapshot'])
assert float(json.loads((metric_model / 'config.json').read_text())['max_depth']) == 80.0
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
log_path = EXP3 / f'training_session_{stamp}.log'
assert not RUN.exists() and not log_path.exists()
cmd = [sys.executable, '-u', str(SOURCE_FULL / 'train.py'),
       '--train-manifest', str(PLAN / 'train.csv'), '--val-manifest', str(PLAN / 'validation.csv'),
       '--rgb-root', str(RGB_LOCAL), '--depth-root', str(DEPTH_LOCAL),
       '--initialization', 'metric_outdoor', '--metric-model', str(metric_model),
       '--max-depth', '65', '--model-max-depth', '80', '--image-size', '518',
       '--batch-size', '1', '--grad-accumulation', '4', '--epochs', '20',
       '--run-kind', 'full', '--encoder-lr', '5e-6', '--weight-decay', '0.01',
       '--workers', '2', '--seed', '42', '--amp', '--save-every-epochs', '1',
       '--keep-recent-checkpoints', '0', '--stop-after-epoch', '5',
       '--memory-preflight-report', str(Path(json.loads(preflight_path.read_text())['report'])), '--output-dir', str(RUN)]
env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}
with log_path.open('xb') as log_stream:
    train_process = subprocess.Popen(cmd, stdout=log_stream, stderr=subprocess.STDOUT, env=env, start_new_session=True)
launch = {'status': 'running', 'pid': train_process.pid, 'run_dir': str(RUN), 'log': str(log_path),
          'epochs_total_for_scheduler': 20, 'stop_after_epoch': 5,
          'initialization': 'original Metric Outdoor Small cache', 'model_output_cap_m': 80,
          'gt_mask_max_m': 65, 'image_size': 518, 'batch_size': 1, 'grad_accumulation': 4,
          'amp': True, 'seed': 42, 'optimizer': 'AdamW', 'encoder_lr': 5e-6, 'head_lr': 5e-5,
          'loss': 'SiLog', 'train_pairs': 384, 'validation_pairs': 128,
          'gpu': torch.cuda.get_device_name(0), 'free_gpu_only': True,
          'compute_units_or_paid_upgrade': False, 'independent_test_used': False,
          'started_utc': datetime.now(timezone.utc).isoformat(), 'command': cmd}
(EXP3 / 'training_launch.json').write_text(json.dumps(launch, ensure_ascii=False, indent=2), encoding='utf-8')
print('5-epoch session started; monitor this process without launching another run.')
print(json.dumps(launch, ensure_ascii=False, indent=2))

## 学習状況を読む

繰り返し実行しても新しい学習を起動しません。ログ、`run_status.json`、`epoch_metrics.jsonl`、checkpoint一覧だけを読みます。5エポックのpause状態になったことと、全checkpointのcomplete markerを確認してください。

In [ ]:
from pathlib import Path
import json, hashlib, psutil
EXP3 = Path('/content/drive/MyDrive/DepthAnythingV2-UAV/experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65')
RUN = Path('/content/drive/MyDrive/DepthAnythingV2-UAV/runs/metric_outdoor_full_20261014_s42_518_cap80_gt65')
launch = json.loads((EXP3 / 'training_launch.json').read_text(encoding='utf-8'))
log = Path(launch['log'])
print('training_process_poll:', train_process.poll() if 'train_process' in globals() else 'handle unavailable; inspect run status/log')
if log.is_file():
    print('log_tail:\n', '\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-50:]))
for name in ('run_status.json', 'latest_checkpoint.json', 'best_checkpoint.json', 'epoch_metrics.jsonl'):
    p = RUN / name
    print(name, p.read_text(encoding='utf-8')[-12000:] if p.is_file() else 'not written yet')
print('complete checkpoints:', sorted(p.name for p in RUN.glob('checkpoint_epoch_*') if (p / 'complete.json').is_file()) if RUN.exists() else [])


In [ ]:
import json, psutil, subprocess
key='metric_outdoor_full_20261014_s42_518_cap80_gt65'
rows=[]
for proc in psutil.process_iter(['pid','cmdline','status','memory_info']):
    cmd=' '.join(proc.info.get('cmdline') or [])
    if 'train.py' in cmd and key in cmd:
        p=psutil.Process(proc.info['pid']); rows.append({'pid':p.pid,'status':p.status(),'rss_bytes':p.memory_info().rss,'cpu_percent_1s':p.cpu_percent(interval=1),'cmd':cmd[:300]})
gpu=subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu','--format=csv,noheader'],capture_output=True,text=True)
print(json.dumps({'training_processes':rows,'nvidia_smi':gpu.stdout.strip(),'nvidia_smi_error':gpu.stderr.strip()},ensure_ascii=False,indent=2))

In [ ]:
from pathlib import Path
import shutil, json
EXP3 = Path('/content/drive/MyDrive/DepthAnythingV2-UAV/experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65')
src = EXP3 / 'training_launch.json'
dst = EXP3 / 'training_launch_attempt_01_preflight_argument_failure.json'
assert src.is_file() and not dst.exists(), '試行1記録が見つからない、または保全先が既に存在します'
shutil.copy2(src, dst)
record = json.loads(dst.read_text(encoding='utf-8'))
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, sys, psutil, torch
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
for name, archived in [('training_launch.json','training_launch_attempt_01_preflight_argument_failure.json'), ('stage_existing_drive_data.log','stage_attempt_01.log'), ('data_stage_verified.json','data_stage_verified_attempt_01.json')]:
    src, dst = EXP3 / name, EXP3 / archived
    if src.is_file() and not dst.exists(): shutil.copy2(src, dst)
sys.path.insert(0, str(EXP3 / 'source_full_v1'))
from runtime_state import dataset_status
PLAN = ROOT / 'uav_metric_finetuning/config/verified_plan'
local = dataset_status(PLAN, Path('/content/uav3dcrop/rgb'), Path('/content/uav3dcrop/depth'), ('train','validation'))
preflight_summary = json.loads((EXP3 / 'gpu_preflight_verified.json').read_text())
report_path = Path(preflight_summary['report'])
report = json.loads(report_path.read_text())
active = []
for proc in psutil.process_iter(['pid','cmdline']):
    cmd = ' '.join(proc.info.get('cmdline') or [])
    if 'train.py' in cmd and 'metric_outdoor_full_20261014_s42_518_cap80_gt65' in cmd: active.append({'pid':proc.info['pid'],'cmd':cmd[:240]})
result = {'checked_utc':datetime.now(timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'cuda_available':torch.cuda.is_available(),'local_dataset':local,'run_dir_exists':RUN.exists(),'active_training_processes':active,'preflight_status':report.get('status'),'preflight_training_feasible':report.get('training_feasible'),'preflight_report':str(report_path),'gpu_preflight':report.get('device'),'model_output_cap_m':report.get('initialization',{}).get('max_depth_m'),'gt_max_depth_m':report.get('gt_max_depth_m'),'preserved_files':[p.name for p in EXP3.glob('*attempt_01*')]}
print(json.dumps(result,ensure_ascii=False,indent=2))
assert torch.cuda.is_available() and local['complete'] and local['complete_pairs']==512 and local['present_files']==1024 and local['planned_bytes']==7850330776
assert not RUN.exists() and not active and report.get('status')=='pass' and report.get('training_feasible') is True

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, sys, psutil, torch
ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
sys.path.insert(0, str(EXP3 / 'source_full_v1'))
from runtime_state import dataset_status
PLAN = ROOT / 'uav_metric_finetuning/config/verified_plan'
local = dataset_status(PLAN, Path('/content/uav3dcrop/rgb'), Path('/content/uav3dcrop/depth'), ('train','validation'))
preflight_summary = json.loads((EXP3 / 'gpu_preflight_verified.json').read_text())
report_path = Path(preflight_summary['report'])
report = json.loads(report_path.read_text())
active = []
for proc in psutil.process_iter(['pid','cmdline']):
    cmd = ' '.join(proc.info.get('cmdline') or [])
    if 'train.py' in cmd and 'metric_outdoor_full_20261014_s42_518_cap80_gt65' in cmd: active.append({'pid':proc.info['pid'],'cmd':cmd[:240]})
result = {'checked_utc':datetime.now(timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'cuda_available':torch.cuda.is_available(),'local_dataset':local,'run_dir_exists':RUN.exists(),'active_training_processes':active,'preflight_status':report.get('status'),'preflight_training_feasible':report.get('training_feasible'),'preflight_report':str(report_path),'gpu_preflight':report.get('device'),'model_output_cap_m':report.get('initialization',{}).get('max_depth_m'),'gt_max_depth_m':report.get('gt_max_depth_m')}
print(json.dumps(result,ensure_ascii=False,indent=2))
assert torch.cuda.is_available() and local['complete'] and local['complete_pairs']==512 and local['present_files']==1024 and local['planned_bytes']==7850330776
assert not RUN.exists() and not active and report.get('status')=='pass' and report.get('training_feasible') is True

## 5エポック終了後：統計表、曲線、報告書をDriveへ保存

学習が `paused_after_epoch` で終了した後に実行してください。全画素のvalidation指標と固定サンプルの形状指標を明確に分け、RMSE最良と20cm以上局所差MAE最良のcheckpointを別々に記録します。モデルCの1エポック結果は破壊・再評価せず、第2回の保存済み実測値を参照線として表示します。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, json, math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
EXP3 = ROOT / 'experiments/metric_outdoor_full_20261014_s42_518_cap80_gt65'
RUN = ROOT / 'runs/metric_outdoor_full_20261014_s42_518_cap80_gt65'
status = json.loads((RUN / 'run_status.json').read_text(encoding='utf-8'))
rows = [json.loads(line) for line in (RUN / 'epoch_metrics.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert status['status'] == 'paused_after_epoch' and status['completed_epochs'] == 5 and len(rows) == 5, (status, len(rows))
assert [row['epoch'] for row in rows] == [1, 2, 3, 4, 5]
baseline_c = {'mae_m': 1.1324, 'rmse_m': 1.7490, 'abs_rel': 0.05208, 'pearson_r': 0.9773,
              'local_depth_delta_mae_m': 0.06800, 'gt_ge_20cm_amplitude_ratio': 0.327,
              'gt_ge_20cm_sign_match_rate': 0.7795}
sample_key = 'all'
def shape_metric(row, subgroup, field):
    return row['validation']['sampled_local_shape']['groups'][subgroup][field]

result_dir = EXP3 / 'results'
result_dir.mkdir(exist_ok=True)
json_path = result_dir / 'epoch_metrics.json'
csv_path = result_dir / 'epoch_metrics.csv'
json_path.write_text(json.dumps({'run_status': status, 'model_c_reference': baseline_c,
                                 'metric_definitions': {'absolute_and_local14': 'all valid validation pixels/pairs',
                                                        'sampled_local_shape': 'fixed seed 42+image_index*10000+14; 100000 candidate pairs per image/axis without replacement; GT-valid pair sample'},
                                 'epochs': rows}, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
columns = ['epoch', 'train_silog', 'validation_mae_m', 'validation_rmse_m', 'validation_abs_rel',
           'validation_pearson_r', 'local14_mae_full_pairs_m', 'sampled_gt_0_5cm_mae_m',
           'sampled_gt_ge_20cm_mae_m', 'sampled_gt_ge_20cm_amplitude_ratio',
           'sampled_gt_ge_20cm_sign_match_rate', 'sampled_edge_0_128px_mae_m',
           'sampled_interior_ge_512px_mae_m', 'epoch_seconds', 'peak_allocated_bytes',
           'peak_reserved_bytes', 'amp_skipped_steps', 'per_crop_full_pixels_json', 'per_view_full_pixels_json']
with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        v = row['validation']
        all_group = v['sampled_local_shape']['groups'][sample_key]
        large = all_group['gt_abs_delta_ge_20cm']
        small = all_group['gt_abs_delta_0_5cm']
        writer.writerow({'epoch': row['epoch'], 'train_silog': row['train_silog'],
                         'validation_mae_m': v['mae_m'], 'validation_rmse_m': v['rmse_m'],
                         'validation_abs_rel': v['abs_rel'], 'validation_pearson_r': v['pearson_r'],
                         'local14_mae_full_pairs_m': v['local_depth_delta_mae_m'],
                         'sampled_gt_0_5cm_mae_m': small['delta_mae_m'],
                         'sampled_gt_ge_20cm_mae_m': large['delta_mae_m'],
                         'sampled_gt_ge_20cm_amplitude_ratio': large['amplitude_ratio'],
                         'sampled_gt_ge_20cm_sign_match_rate': large['sign_match_rate'],
                         'sampled_edge_0_128px_mae_m': shape_metric(row, 'border_0_128px', 'all_sampled_pairs')['delta_mae_m'],
                         'sampled_interior_ge_512px_mae_m': shape_metric(row, 'interior_ge_512px', 'all_sampled_pairs')['delta_mae_m'],
                         'epoch_seconds': row['epoch_seconds'], 'peak_allocated_bytes': row['peak_allocated_bytes'],
                         'peak_reserved_bytes': row['peak_reserved_bytes'], 'amp_skipped_steps': row['amp_skipped_steps'],
                         'per_crop_full_pixels_json': json.dumps(v['per_crop'], ensure_ascii=False),
                         'per_view_full_pixels_json': json.dumps(v['per_view'], ensure_ascii=False)})

plot_specs = [
    ('validation_rmse.png', 'validation RMSE (m)', 'rmse_m', baseline_c['rmse_m']),
    ('validation_abs_rel.png', 'validation AbsRel', 'abs_rel', baseline_c['abs_rel']),
    ('local14_mae.png', '14px local depth-difference MAE (m, all valid pairs)', 'local_depth_delta_mae_m', baseline_c['local_depth_delta_mae_m']),
    ('gt_ge20cm_amplitude_ratio.png', 'GT |delta| >= 20 cm: predicted / GT amplitude (fixed sample)', 'amplitude_ratio', baseline_c['gt_ge_20cm_amplitude_ratio']),
    ('gt_ge20cm_sign_match.png', 'GT |delta| >= 20 cm: sign agreement (fixed sample)', 'sign_match_rate', baseline_c['gt_ge_20cm_sign_match_rate']),
]
for filename, ylabel, key, reference in plot_specs:
    values = []
    for row in rows:
        if key == 'amplitude_ratio':
            value = row['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm'][key]
        elif key == 'sign_match_rate':
            value = row['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm'][key]
        else:
            value = row['validation'][key]
        values.append(value)
    fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
    ax.plot([r['epoch'] for r in rows], values, marker='o', linewidth=2, label='full run')
    ax.axhline(reference, color='#d55e00', linestyle='--', label='trial2 Model C (1 epoch)')
    ax.set(xlabel='epoch', ylabel=ylabel, xticks=[r['epoch'] for r in rows])
    ax.grid(alpha=0.25)
    ax.legend()
    ax.set_title('Metric Outdoor Small; total schedule = 20 epochs')
    fig.savefig(result_dir / filename, dpi=160)
    plt.close(fig)

best_rmse = min(rows, key=lambda r: r['validation']['rmse_m'])
best_shape = min(rows, key=lambda r: r['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm']['delta_mae_m'])
(RUN / 'best_shape_checkpoint.json').write_text(json.dumps({'path': f"checkpoint_epoch_{best_shape['epoch']:04d}",
    'epoch': best_shape['epoch'], 'selection_metric': 'minimum sampled GT |delta| >= 20cm local delta MAE',
    'metric_value_m': best_shape['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm']['delta_mae_m'],
    'sample_count': best_shape['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm']['pair_count'],
    'split': 'validation only'}, ensure_ascii=False, indent=2), encoding='utf-8')

report = ['# 第3回 Metric Outdoor Small 学習（5/20 epoch）', '',
          '20epoch総期間のPolynomialLRを維持し、epoch 5で安全停止した。初期重みはDrive上の元Metric Outdoor Small。モデル出力上限80m、GT有効上限65m。Independent testと旧52組は未使用。', '',
          '|epoch|train SiLog|val MAE (m)|val RMSE (m)|AbsRel|Pearson|14px全ペア局所MAE (m)|20cm+標本局所MAE (m)|振幅比|符号一致|時間 (s)|peak allocated/reserved (GiB)|AMP skip|',
          '|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for row in rows:
    v = row['validation']; g = v['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm']
    report.append(f"|{row['epoch']}|{row['train_silog']:.6f}|{v['mae_m']:.4f}|{v['rmse_m']:.4f}|{v['abs_rel']:.5f}|{v['pearson_r']:.4f}|{v['local_depth_delta_mae_m']:.5f}|{g['delta_mae_m']:.5f}|{g['amplitude_ratio']:.4f}|{g['sign_match_rate']:.4f}|{row['epoch_seconds']:.1f}|{row['peak_allocated_bytes']/1024**3:.3f}/{row['peak_reserved_bytes']/1024**3:.3f}|{row['amp_skipped_steps']}|")
report += ['', '## checkpoint選択', '',
           f"- RMSE最良: epoch {best_rmse['epoch']}（{best_rmse['validation']['rmse_m']:.4f} m）, `{(RUN / 'best_checkpoint.json').name}`.",
           f"- 20cm以上の局所差MAE最良: epoch {best_shape['epoch']}（{best_shape['validation']['sampled_local_shape']['groups'][sample_key]['gt_abs_delta_ge_20cm']['delta_mae_m']:.5f} m）, `best_shape_checkpoint.json`.",
           '- 20cm以上の標本は前回と同じseed・候補座標ルールで抽出し、全ペア指標と分けて保存した。', '',
           '## 作物・方向・画像端', '',
           '各epochの `epoch_metrics.json` に作物別・撮影方向別の全有効画素指標と、画像端0–128px・内部512px以上の固定標本局所指標を記録した。主評価は全画面のまま。', '',
           '## 第2回Model Cとの参照', '',
           '保存済み第2回Model C: RMSE 1.7490m、AbsRel 0.05208、14px局所差MAE 0.06800m、20cm+振幅比0.327、符号一致率0.7795。第3回の初期モデルは元の未追加学習Metric Outdoor Smallから開始し、testは使っていない。', '',
           '## 保存物', '',
           f'- epoch CSV: `{csv_path}`', f'- epoch JSON: `{json_path}`',
           '- 学習曲線: results/validation_rmse.png, validation_abs_rel.png, local14_mae.png, gt_ge20cm_amplitude_ratio.png, gt_ge20cm_sign_match.png',
           f'- checkpoint: `{RUN}`（epoch 0–5のcomplete markerとSHA256を検証）',
           '- 再開: 同じrun、epochs=20、同じコード・manifest・学習設定を保ち、`--resume --stop-after-epoch 10`。scheduler期間を5へ変更しない。',
           '- 次の15epochは自動開始していない。無料GPUを再割当でき、validationの絶対・局所形状の推移が継続価値を示す場合に、同一runから再開する。', '',
           f"生成時刻: {datetime.now(timezone.utc).isoformat()}"]
report_path = EXP3 / 'report_ja.md'
report_path.write_text('\n'.join(report) + '\n', encoding='utf-8')

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()
verified = {}
for checkpoint in sorted(RUN.glob('checkpoint_epoch_*')):
    marker_path = checkpoint / 'complete.json'
    if not marker_path.is_file():
        continue
    marker = json.loads(marker_path.read_text(encoding='utf-8'))
    for filename, expected in marker['file_sha256'].items():
        assert sha(checkpoint / filename) == expected, f'checkpoint SHA mismatch: {checkpoint / filename}'
    verified[checkpoint.name] = marker
assert set(verified) == {f'checkpoint_epoch_{i:04d}' for i in range(6)}, verified.keys()
assert len(list(result_dir.glob('*.png'))) == 5
sync_receipt = {'status': 'pass', 'checkpoints_sha256_verified': sorted(verified),
                'metrics_json': str(json_path), 'metrics_csv': str(csv_path),
                'report': str(report_path), 'figure_paths': [str(p) for p in sorted(result_dir.glob('*.png'))],
                'drive_free_bytes': __import__('shutil').disk_usage(ROOT).free,
                'completed_utc': datetime.now(timezone.utc).isoformat()}
(EXP3 / 'sync_receipt.json').write_text(json.dumps(sync_receipt, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(sync_receipt, ensure_ascii=False, indent=2))
print(report_path.read_text(encoding='utf-8'))